# 05 — Feature Engineering

Create derived features and produce the final modelling-ready dataset.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yaml, os

with open('../configs/paths.yaml') as f:
    paths = yaml.safe_load(f)
with open('../configs/config.yaml') as f:
    cfg = yaml.safe_load(f)

plt.rcParams.update({'figure.facecolor':'#0f172a','axes.facecolor':'#1e293b',
    'axes.edgecolor':'#334155','axes.labelcolor':'#e2e8f0',
    'xtick.color':'#94a3b8','ytick.color':'#94a3b8','text.color':'#e2e8f0'})
PALETTE = ['#38bdf8','#fb7185','#34d399','#fbbf24','#a78bfa','#f97316']

df = pd.read_csv(f'../{paths["data"]["raw"]}')
df['dteday'] = pd.to_datetime(df['dteday'], dayfirst=True)
DROP_COLS = cfg['features']['drop_columns'] + ['dteday']
df = df.drop(columns=[c for c in DROP_COLS if c in df.columns])
print('Base shape:', df.shape)

In [ ]:
# ── Feature 1: Comfort Index (temp − hum interaction) ─────────────────────────
df['comfort_index'] = df['temp'] - 0.5 * df['hum'] / 100
print('comfort_index: min={:.3f}, max={:.3f}'.format(df['comfort_index'].min(), df['comfort_index'].max()))

In [ ]:
# ── Feature 2: Weekend flag ───────────────────────────────────────────────────
df['is_weekend'] = (df['weekday'].isin([0, 6])).astype(int)
print('is_weekend distribution:', df['is_weekend'].value_counts().to_dict())

In [ ]:
# ── Feature 3: Quarter ────────────────────────────────────────────────────────
df['quarter'] = pd.cut(df['mnth'], bins=[0,3,6,9,12],
                        labels=[1,2,3,4]).astype(int)
print('Quarter distribution:', df['quarter'].value_counts().sort_index().to_dict())

In [ ]:
# ── Feature 4: Warm season flag ───────────────────────────────────────────────
df['is_warm_season'] = (df['season'].isin([2, 3])).astype(int)  # Summer=2, Fall=3
print('is_warm_season:', df['is_warm_season'].value_counts().to_dict())

In [ ]:
# ── Feature 5: Bad weather flag ───────────────────────────────────────────────
df['is_bad_weather'] = (df['weathersit'] >= 3).astype(int)
print('is_bad_weather:', df['is_bad_weather'].value_counts().to_dict())

In [ ]:
# ── Feature 6: temp × yr interaction (growth trend × season) ─────────────────
df['temp_yr'] = df['temp'] * df['yr']
print('temp_yr sample:', df['temp_yr'].describe().round(3).to_dict())

In [ ]:
# ── Show final feature set ────────────────────────────────────────────────────
print('Final columns:', df.columns.tolist())
print('Shape:', df.shape)
df.head()

In [ ]:
# ── Correlations of new features with cnt ────────────────────────────────────
new_feats = ['comfort_index','is_weekend','quarter','is_warm_season','is_bad_weather','temp_yr']
print('Correlation of new features with cnt:')
print(df[new_feats + ['cnt']].corr()['cnt'].drop('cnt').sort_values(ascending=False).to_string())

In [ ]:
# ── Visual: new feature importance ───────────────────────────────────────────
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression

X = df.drop(columns=['cnt'])
y = df['cnt']
scaler = StandardScaler()
X_s = scaler.fit_transform(X)

lr = LinearRegression().fit(X_s, y)
coefs = pd.Series(np.abs(lr.coef_), index=X.columns).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(12, 5))
ax.barh(coefs.index[::-1], coefs.values[::-1], color=PALETTE[0])
ax.set(title='|Coefficient| from StandardScaled Linear Regression', xlabel='|Coefficient|')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('../outputs/feature_importance_linear.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Save engineered dataset ───────────────────────────────────────────────────
os.makedirs(f'../{paths["data"]["processed"]}', exist_ok=True)
out_path = f'../{paths["data"]["processed"]}data_engineered.csv'
df.to_csv(out_path, index=False)
print(f'Saved → {out_path}  |  shape: {df.shape}')

## Feature Engineering Summary

| Feature | Description | Expected Effect |
|---|---|---|
| `comfort_index` | temp − 0.5×(hum/100) | Higher comfort → higher demand |
| `is_weekend` | 1 if Sat/Sun | Captures leisure vs commute |
| `quarter` | 1–4 | Coarser seasonality signal |
| `is_warm_season` | 1 if Summer/Fall | Strong demand booster |
| `is_bad_weather` | 1 if weathersit ≥ 3 | Demand suppressor |
| `temp_yr` | temp × yr | Growth trend + temperature joint effect |

**Next:** `06_Model_Building.ipynb`
